In [1]:
import src.database.scripts.sql as sql
from src.database.scrappers.item_data_fetch import item_data_fetch

In [ ]:
def drop_item_data_table():
    query = "DROP TABLE IF EXISTS season_8.item_data"
    cursor.execute(query)


def create_item_data_table():
    columns = body_list[0].keys()
    type_map = {}
    for col in columns:
        for row in body_list:
            col_type = type(row[col])
            if col_type != type(None):
                if col_type == int:
                    type_map[col] = "INT"
                if col_type == float:
                    type_map[col] = "FLOAT"            
                if col_type == str:
                    type_map[col] = "TEXT"
                break
        else:
            type_map[col] = "TEXT"
    query_col_text = ", ".join(f"{col} {dtype}" for col, dtype in type_map.items())
    
    query = f"""
        CREATE TABLE IF NOT EXISTS season_8.item_data ({query_col_text}
        )"""
    cursor.execute(query)
    
    query = f"""
        ALTER TABLE season_8.item_data
        ADD CONSTRAINT unique_item_id UNIQUE (id);
        """
    cursor.execute(query)


def insert_item_data():
    columns = body_list[0].keys()
    rows = [tuple(row[col] for col in columns) for row in body_list]

    query_col_text = ", ".join(columns)
    row_placeholder = ", ".join(["%s"] * len(columns))
    
    query = f"""
    INSERT INTO season_8.item_data ({query_col_text})
    VALUES ({row_placeholder})
    """
    cursor.executemany(query, rows)

In [ ]:
conn = sql.connect_pc()
cursor = conn.cursor()

print("dropping existing table...")
drop_item_data_table()

print("fetching data...")
body_list = item_data_fetch()

print("creating table...")
create_item_data_table()

print("inserting data...")
insert_item_data()

conn.commit()
conn.close()
print("compelete!")

dropping existing table...
fetching data...
creating table...
inserting data...
